# Multiclass Logistic Regression from Scratch
## Experiments on PlantVillage and PlantDoc Datasets

Bu notebook, daha önce ön işlenmiş (preprocessed) PlantVillage ve PlantDoc verilerini
yükler ve sıfırdan yazılmış Logistic Regression (Softmax) modelini her iki veri seti
üzerinde ayrı ayrı eğitir. Sonrasında accuracy, precision, recall, F1 ve confusion
matrix metrikleri hesaplanır.


## Hücre 1 – Import ve temel ayarlar


In [ ]:
import numpy as np
import math
import random


## Hücre 2 – Logistic Regression yardımcı fonksiyonları


In [ ]:
def softmax(logits):
    max_logit = max(logits)
    exps = [math.exp(z - max_logit) for z in logits]
    s = sum(exps)
    if s == 0.0:
        c = len(logits)
        return [1.0 / c for _ in range(c)]
    return [e / s for e in exps]


def cross_entropy_loss(probs, true_class):
    eps = 1e-15
    p = probs[true_class]
    p = max(min(p, 1.0 - eps), eps)
    return -math.log(p)


## Hücre 3 – MulticlassLogisticRegression sınıfı


In [ ]:
class MulticlassLogisticRegression:
    def __init__(self, num_features, num_classes, learning_rate=0.1):
        self.num_features = num_features
        self.num_classes = num_classes
        self.learning_rate = learning_rate

        self.W = [
            [(random.random() - 0.5) * 0.01 for _ in range(num_classes)]
            for _ in range(num_features)
        ]
        self.b = [(random.random() - 0.5) * 0.01 for _ in range(num_classes)]

    def _compute_logits(self, x):
        logits = [0.0 for _ in range(self.num_classes)]
        for k in range(self.num_classes):
            s = 0.0
            for j in range(self.num_features):
                s += x[j] * self.W[j][k]
            s += self.b[k]
            logits[k] = s
        return logits

    def predict_proba_one(self, x):
        logits = self._compute_logits(x)
        probs = softmax(logits)
        return probs

    def predict_one(self, x):
        probs = self.predict_proba_one(x)
        best_class = 0
        best_prob = probs[0]
        for k in range(1, self.num_classes):
            if probs[k] > best_prob:
                best_prob = probs[k]
                best_class = k
        return best_class

    def predict(self, X):
        return [self.predict_one(x) for x in X]

    def fit(self, X, y, num_epochs=10, shuffle=True, verbose=True,
            X_val=None, y_val=None):
        n_samples = len(X)

        for epoch in range(num_epochs):
            indices = list(range(n_samples))
            if shuffle:
                random.shuffle(indices)

            total_loss = 0.0
            correct = 0

            progress_step = max(1, n_samples // 10)

            for step, idx in enumerate(indices):
                x = X[idx]
                true_class = y[idx]

                logits = self._compute_logits(x)
                probs = softmax(logits)
                loss = cross_entropy_loss(probs, true_class)
                total_loss += loss

                pred_class = 0
                best_prob = probs[0]
                for k in range(1, self.num_classes):
                    if probs[k] > best_prob:
                        best_prob = probs[k]
                        pred_class = k
                if pred_class == true_class:
                    correct += 1

                for k in range(self.num_classes):
                    if k == true_class:
                        error_k = probs[k] - 1.0
                    else:
                        error_k = probs[k]

                    for j in range(self.num_features):
                        grad_w_jk = error_k * x[j]
                        self.W[j][k] -= self.learning_rate * grad_w_jk

                    grad_b_k = error_k
                    self.b[k] -= self.learning_rate * grad_b_k

                if verbose and (step + 1) % progress_step == 0:
                    pct = (step + 1) / n_samples * 100.0
                    print(
                        f"Epoch {epoch + 1}/{num_epochs} - "
                        f"{pct:5.1f}% completed",
                        end="\r",
                        flush=True
                    )

            avg_loss = total_loss / n_samples
            train_acc = correct / n_samples

            val_acc = None
            if X_val is not None and y_val is not None and len(X_val) > 0:
                correct_val = 0
                for x_val, y_true_val in zip(X_val, y_val):
                    y_pred_val = self.predict_one(x_val)
                    if y_pred_val == y_true_val:
                        correct_val += 1
                val_acc = correct_val / len(X_val)

            if verbose:
                if val_acc is not None:
                    print(
                        f"Epoch {epoch + 1}/{num_epochs} - "
                        f"loss: {avg_loss:.4f} - "
                        f"train_acc: {train_acc:.4f} - "
                        f"val_acc: {val_acc:.4f}          "
                    )
                else:
                    print(
                        f"Epoch {epoch + 1}/{num_epochs} - "
                        f"loss: {avg_loss:.4f} - "
                        f"train_acc: {train_acc:.4f}          "
                    )
    def save(self, path):
        """
        Save model parameters (W, b, num_features, num_classes)
        to a .npz file.
        """
        W_array = np.array(self.W, dtype=np.float32)
        b_array = np.array(self.b, dtype=np.float32)
        np.savez(
            path,
            W=W_array,
            b=b_array,
            num_features=self.num_features,
            num_classes=self.num_classes,
        )
        print(f"Model saved to {path}")

    @classmethod
    def load(cls, path):
        """
        Load model parameters from a .npz file and return a new instance.
        """
        data = np.load(path)

        num_features = int(data["num_features"])
        num_classes = int(data["num_classes"])

        model = cls(
            num_features=num_features,
            num_classes=num_classes,
            learning_rate=0.01  # lr is irrelevant if we only use for prediction
        )

        W_array = data["W"]
        b_array = data["b"]

        model.W = W_array.tolist()
        model.b = b_array.tolist()

        print(f"Model loaded from {path}")
        print("num_features:", num_features)
        print("num_classes:", num_classes)

        return model

## Hücre 4 – Metrik fonksiyonları (accuracy, confusion matrix, precision, recall, F1)


In [ ]:
def accuracy_score(y_true, y_pred):
    assert len(y_true) == len(y_pred)
    correct = sum(int(t == p) for t, p in zip(y_true, y_pred))
    return correct / len(y_true)


def confusion_matrix(y_true, y_pred, num_classes):
    mat = [[0 for _ in range(num_classes)] for _ in range(num_classes)]
    for t, p in zip(y_true, y_pred):
        mat[t][p] += 1
    return mat


def precision_recall_f1_per_class(y_true, y_pred, num_classes):
    cm = confusion_matrix(y_true, y_pred, num_classes)
    precisions = []
    recalls = []
    f1s = []

    for c in range(num_classes):
        tp = cm[c][c]
        fp = sum(cm[r][c] for r in range(num_classes) if r != c)
        fn = sum(cm[c][k] for k in range(num_classes) if k != c)

        if tp + fp == 0:
            precision = 0.0
        else:
            precision = tp / (tp + fp)

        if tp + fn == 0:
            recall = 0.0
        else:
            recall = tp / (tp + fn)

        if precision + recall == 0:
            f1 = 0.0
        else:
            f1 = 2 * precision * recall / (precision + recall)

        precisions.append(precision)
        recalls.append(recall)
        f1s.append(f1)

    return precisions, recalls, f1s


def macro_average(values):
    if len(values) == 0:
        return 0.0
    return sum(values) / len(values)


## Hücre 5 – PlantVillage verisini yükle, modeli eğit


In [ ]:
# ----- PlantVillage -----
SAVE_DIR_PV = "preprocessed_plantvillage"

X_train_pv = np.load(f"{SAVE_DIR_PV}/train_X.npy")
y_train_pv = np.load(f"{SAVE_DIR_PV}/train_y.npy")

X_val_pv = np.load(f"{SAVE_DIR_PV}/val_X.npy")
y_val_pv = np.load(f"{SAVE_DIR_PV}/val_y.npy")

X_test_pv = np.load(f"{SAVE_DIR_PV}/test_X.npy")
y_test_pv = np.load(f"{SAVE_DIR_PV}/test_y.npy")

print("PlantVillage shapes:")
print("Train:", X_train_pv.shape, y_train_pv.shape)
print("Val:", X_val_pv.shape, y_val_pv.shape)
print("Test:", X_test_pv.shape, y_test_pv.shape)

num_features_pv = X_train_pv.shape[1]
num_classes_pv = len(set(y_train_pv.tolist()))
print("PlantVillage - num_features:", num_features_pv)
print("PlantVillage - num_classes:", num_classes_pv)

X_train_pv_list = X_train_pv.tolist()
y_train_pv_list = y_train_pv.tolist()
X_val_pv_list = X_val_pv.tolist()
y_val_pv_list = y_val_pv.tolist()
X_test_pv_list = X_test_pv.tolist()
y_test_pv_list = y_test_pv.tolist()

learning_rate_pv = 0.01
num_epochs_pv = 20

model_pv = MulticlassLogisticRegression(
    num_features=num_features_pv,
    num_classes=num_classes_pv,
    learning_rate=learning_rate_pv
)

model_pv.fit(
    X_train_pv_list,
    y_train_pv_list,
    num_epochs=num_epochs_pv,
    verbose=True,
    X_val=X_val_pv_list,
    y_val=y_val_pv_list
)


### PlantVillage modeli için metrikleri hesapla


In [ ]:
y_train_pv_pred = model_pv.predict(X_train_pv_list)
y_val_pv_pred = model_pv.predict(X_val_pv_list)
y_test_pv_pred = model_pv.predict(X_test_pv_list)

print("PlantVillage Accuracy:")
print("Train:", accuracy_score(y_train_pv_list, y_train_pv_pred))
print("Val:  ", accuracy_score(y_val_pv_list, y_val_pv_pred))
print("Test: ", accuracy_score(y_test_pv_list, y_test_pv_pred))

cm_pv = confusion_matrix(y_test_pv_list, y_test_pv_pred, num_classes_pv)
prec_pv, rec_pv, f1_pv = precision_recall_f1_per_class(
    y_test_pv_list, y_test_pv_pred, num_classes_pv
)

print("PlantVillage Macro Precision:", macro_average(prec_pv))
print("PlantVillage Macro Recall:   ", macro_average(rec_pv))
print("PlantVillage Macro F1:       ", macro_average(f1_pv))


## Hücre 6 – PlantDoc verisini yükle, modeli eğit


In [ ]:
# ----- PlantDoc -----
SAVE_DIR_PD = "preprocessed_plantdoc"

X_train_pd = np.load(f"{SAVE_DIR_PD}/train_X.npy")
y_train_pd = np.load(f"{SAVE_DIR_PD}/train_y.npy")

X_val_pd = np.load(f"{SAVE_DIR_PD}/val_X.npy")
y_val_pd = np.load(f"{SAVE_DIR_PD}/val_y.npy")

X_test_pd = np.load(f"{SAVE_DIR_PD}/test_X.npy")
y_test_pd = np.load(f"{SAVE_DIR_PD}/test_y.npy")

print("PlantDoc shapes:")
print("Train:", X_train_pd.shape, y_train_pd.shape)
print("Val:", X_val_pd.shape, y_val_pd.shape)
print("Test:", X_test_pd.shape, y_test_pd.shape)

num_features_pd = X_train_pd.shape[1]
num_classes_pd = len(set(y_train_pd.tolist()))
print("PlantDoc - num_features:", num_features_pd)
print("PlantDoc - num_classes:", num_classes_pd)

X_train_pd_list = X_train_pd.tolist()
y_train_pd_list = y_train_pd.tolist()
X_val_pd_list = X_val_pd.tolist()
y_val_pd_list = y_val_pd.tolist()
X_test_pd_list = X_test_pd.tolist()
y_test_pd_list = y_test_pd.tolist()

learning_rate_pd = 0.01
num_epochs_pd = 20

model_pd = MulticlassLogisticRegression(
    num_features=num_features_pd,
    num_classes=num_classes_pd,
    learning_rate=learning_rate_pd
)

model_pd.fit(
    X_train_pd_list,
    y_train_pd_list,
    num_epochs=num_epochs_pd,
    verbose=True,
    X_val=X_val_pd_list,
    y_val=y_val_pd_list
)


### PlantDoc modeli için metrikleri hesapla


In [ ]:
y_train_pd_pred = model_pd.predict(X_train_pd_list)
y_val_pd_pred = model_pd.predict(X_val_pd_list)
y_test_pd_pred = model_pd.predict(X_test_pd_list)

print("PlantDoc Accuracy:")
print("Train:", accuracy_score(y_train_pd_list, y_train_pd_pred))
print("Val:  ", accuracy_score(y_val_pd_list, y_val_pd_pred))
print("Test: ", accuracy_score(y_test_pd_list, y_test_pd_pred))

cm_pd = confusion_matrix(y_test_pd_list, y_test_pd_pred, num_classes_pd)
prec_pd, rec_pd, f1_pd = precision_recall_f1_per_class(
    y_test_pd_list, y_test_pd_pred, num_classes_pd
)

print("PlantDoc Macro Precision:", macro_average(prec_pd))
print("PlantDoc Macro Recall:   ", macro_average(rec_pd))
print("PlantDoc Macro F1:       ", macro_average(f1_pd))


### Modelleri Kaydedelim


In [ ]:
import os

os.makedirs("models", exist_ok=True)  # create folder if it doesn't exist

# Save PlantVillage model
model_pv.save("models/logreg_plantvillage.npz")

# Save PlantDoc model
model_pd.save("models/logreg_plantdoc.npz")
